<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/IMDBGPT_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [4]:
import transformers
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
from transformers import AutoTokenizer
from datasets import load_dataset

In [ ]:
raw_dataset = load_dataset('imdb')
train_Data = raw_dataset['train']
val_Data = raw_dataset['test']

In [ ]:
autoToken = AutoTokenizer.from_pretrained('bert-base-uncased')
autoToken.vocab_size

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)
print("GPU Count:", torch.cuda.device_count())

Device: cpu
GPU Count: 0


In [24]:
max_len = 256
vocab_size = autoToken.vocab_size

In [8]:
trained_data = autoToken(text=list(train_Data['text']),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
validation_data = autoToken(text=list(val_Data['text']),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [19]:
class IMDBDataSet(Dataset):
  def __init__(self,encoding,labels):
    self.encoding = encoding
    self.labels = labels

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    # 1. Grab the full sequence of token IDs and the attention mask
    full_ids = self.encoding['input_ids'][idx]
    full_mask = self.encoding['attention_mask'][idx]
    return {
        'input_ids':full_ids[:-1],
        'attention_mask':self.encoding['attention_mask'][idx],
        'target_ids':full_mask[:-1]
    }

In [20]:
train_dataset = IMDBDataSet(encoding=trained_data,labels=train_Data['label'])
val_dataset = IMDBDataSet(encoding=validation_data,labels=val_Data['label'])

In [21]:
train_Ds = DataLoader(dataset=train_dataset,batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
val_Ds = DataLoader(dataset=val_dataset,batch_size=64,shuffle=False,pin_memory=True,num_workers=2)

By default, self-attention looks at the entire sentence at once (like BERT). When you pass your triangular matrix into attn_mask, you are forcing it into Decoder mode. It tells the layer: "Follow this specific map of what is allowed to be seen."

In [12]:
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,ff_dim) -> None:
    super().__init__()
    self.attention = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads,dropout=0.1)
    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim,out_features=ff_dim),
        nn.GELU(),
        nn.Linear(in_features=ff_dim,out_features=embed_dim),
        nn.Dropout(0.1)
    )

    self.layernorm1 = nn.LayerNorm(embed_dim)
    self.layernorm2 = nn.LayerNorm(embed_dim)

  def forward(self,x,causal_mask):
    att,_ = self.attention(query=x, key=x, value=x, attn_mask=causal_mask)
    x = self.layernorm1(x + att)
    mlpOutput = self.mlp(x)
    return self.layernorm2(att + x)

In [13]:
class WordEmbedding(nn.Module):
  def __init__(self,embed_dim) -> None:
    super().__init__()
    self.wordEmbed = nn.Embedding(num_embeddings=vocab_size,embedding_dim=embed_dim)

  def forward(self,x):
    word = self.wordEmbed(x)
    return word

In [17]:
class NLPTransformer(nn.Module):
  def __init__(self,embed_dim,max_len,num_heads,ff_dim,num_layers,vocab_size) -> None:
    super().__init__()
    self.wordEmbed = WordEmbedding(embed_dim=embed_dim)
    self.positionalEmbed = nn.Parameter(torch.zeros(1,max_len,embed_dim))

    self.transformer_layers = nn.ModuleList([
        TransformerBlock(embed_dim=embed_dim,num_heads=num_heads,ff_dim=ff_dim)
        for _ in range(num_layers)
    ])

    self.dropout = nn.Dropout(0.2)
    self.classifier = nn.Linear(in_features=embed_dim,out_features=vocab_size) #  The Output Mouth: Projects to vocab_size for next-word classification

  def generate_causal_mask(self,seq_len,device):
    #create triangular sheild
    mask = torch.triu(torch.ones(seq_len,seq_len,device=device),diagonal=1)
    return mask.masked_fill(mask == 1,float('-inf')) # -inf blocks the future words

  def forward(self,x,mask):
    seq_len = x.shape[1]
    x = self.wordEmbed(x) + self.positionalEmbed[:,:seq_len,:]
    x = self.dropout(x)

    #create a shield dynamically
    causal_mask = self.generate_causal_mask(seq_len,x.device)

    for transformer_layer in self.transformer_layers:
      x = transformer_layer(x, causal_mask=causal_mask)

    # 3. NO POOLING HERE! We keep the timeline intact.
    # Output shape: [Batch, Seq_Len, Vocab_Size]
    return self.classifier(x)

In [26]:
# 1. Initialize your custom engine from scratch
model = NLPTransformer(embed_dim=256,
                       max_len=200,
                       num_heads=4,
                       ff_dim= 1024,
                       num_layers=8,
                       vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        # Unpack your pre-tokenized inputs and shifted targets
        input_ids = batch['input_ids'].to(device)    # Words 0 to N-1
        targets = batch['target_ids'].to(device)    # Words 1 to N (The Answers)

        optimizer.zero_grad()

        # 1. Forward Pass -> outputs shape: [Batch, Seq_Len, Vocab_Size]
        outputs = model(input_ids)

        # 2. CRITICAL FIX: Flatten 3D to 2D for CrossEntropyLoss
        # View outputs as: [Batch * Seq_Len, Vocab_Size]
        # View targets as: [Batch * Seq_Len]
        loss = loss_fn(outputs.view(-1, outputs.size(-1)), targets.view(-1))

        # 3. Backprop math stays the same
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # 4. Calculate Perplexity (The metric for writers)
    avg_loss = total_loss / len(train_loader)
    perplexity = torch.exp(torch.tensor(avg_loss))

    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Perplexity: {perplexity:.2f}")

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50):
    model.eval() # Set to evaluation mode

    # 1. Turn your prompt into numbers
    # (Just like we did in the Dataset)
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    for _ in range(max_new_tokens):
        # 2. Get the model's classification guesses
        with torch.no_grad():
            outputs = model(input_ids)

        # 3. Take only the VERY LAST word's guess
        # Shape of outputs is [1, Seq_Len, Vocab_Size]
        next_token_logits = outputs[:, -1, :]

        # 4. Pick the winner (The word with the highest score)
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

        # 5. Glue the new word to the end of the sentence
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        # 6. Stop if the model writes the 'End of Sentence' token
        if next_token.item() == tokenizer.sep_token_id:
            break

    # 7. Turn the numbers back into English words
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


In [ ]:
# 1. Choose your 'Seed' (The prompt)
my_prompt = "The cinematography in this film was"

# 2. Call the generator
generated_review = generate(model, autotoken, my_prompt, max_new_tokens=30)

# 3. See the magic
print(generated_review)
